In [11]:
%pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: C:\Users\fisuk\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


In [12]:
import requests
from pprint import pprint
import pandas as pd
from dotenv import load_dotenv
import os

In [15]:
load_dotenv()

True

In [16]:
CLIENT_ID = os.getenv("RAMP_CLIENT_ID")
CLIENT_SECRET = os.getenv("RAMP_CLIENT_SECRET")

In [18]:
if not CLIENT_ID or not CLIENT_SECRET:
    raise RuntimeError("Set RAMP_CLIENT_ID and RAMP_CLIENT_SECRET")

BASE_URL = "https://api.ramp.com/developer/v1"
TOKEN_URL = "https://api.ramp.com/developer/v1/token" 

In [31]:
import time
import base64

_access_token = None
_expires_at = 0

def get_access_token():
    global _access_token, _expires_at

    if _access_token and time.time() < _expires_at:
        return _access_token

    # Build HTTP Basic auth header
    basic_auth = base64.b64encode(
        f"{CLIENT_ID}:{CLIENT_SECRET}".encode()
    ).decode()

    resp = requests.post(
        TOKEN_URL,
        headers={
            "Authorization": f"Basic {basic_auth}",
            "Content-Type": "application/x-www-form-urlencoded",
            "Accept": "application/json",
        },
        data={
            "grant_type": "client_credentials",
            "scope": "transactions:read vendors:read users:read accounting:read bills:read accounting:write",
        },
    )

    # Helpful debugging if Ramp returns errors
    if resp.status_code != 200:
        raise RuntimeError(f"Token error {resp.status_code}: {resp.text}")

    token = resp.json()
    _access_token = token["access_token"]
    _expires_at = time.time() + token.get("expires_in", 3600) - 30

    return _access_token


In [32]:
def ramp_headers():
    return {
        "Authorization": f"Bearer {get_access_token()}",
        "Content-Type": "application/json",
        "Accept": "application/json",
    }

def ramp_get(endpoint, params=None):
    url = f"{BASE_URL}{endpoint}"
    resp = requests.get(url, headers=ramp_headers(), params=params)
    resp.raise_for_status()
    return resp.json()

def ramp_post(endpoint, payload=None):
    url = f"{BASE_URL}{endpoint}"
    resp = requests.post(url, headers=ramp_headers(), json=payload)
    resp.raise_for_status()
    return resp.json()


In [33]:
# Transactions
acct_fields = ramp_get("/accounting/fields")
pprint(acct_fields)

{'data': [{'accounting_connection_id': '364a2b18-3ad2-41db-9f96-5f82ab5738ac',
           'created_at': '2026-01-14T16:08:20+00:00',
           'display_name': None,
           'id': 'Vendor Class',
           'input_type': 'SINGLE_CHOICE',
           'is_active': True,
           'is_required_for': [],
           'is_splittable': True,
           'name': 'Vendor Class',
           'provider_name': 'ACCOUNTING_SEED',
           'ramp_id': 'b47f80c0-b72e-475a-8bc7-e6c44b44ca29',
           'updated_at': '2026-01-14T16:08:20+00:00'},
          {'accounting_connection_id': '364a2b18-3ad2-41db-9f96-5f82ab5738ac',
           'created_at': '2026-01-14T15:31:17+00:00',
           'display_name': None,
           'id': 'Conflict of Interest',
           'input_type': 'SINGLE_CHOICE',
           'is_active': True,
           'is_required_for': [],
           'is_splittable': True,
           'name': 'Conflict of Interest',
           'provider_name': 'ACCOUNTING_SEED',
           'ramp_id': 'd6

In [34]:
acct_update_endpoint = "https://api.ramp.com/developer/v1/accounting/field-options"

In [35]:
payload = {
  "accounting_connection_id": '364a2b18-3ad2-41db-9f96-5f82ab5738ac',
  "field_id": '54439799-68cc-4c36-8bcd-233557b6ea70',
    'input_type': 'SINGLE_CHOICE'
  "options": [
    {
      "id": "001",
      "value": "Community Meetings"
    },
    {
      "id": "002",
      "value": "Site visits"
    }
  ]
}